In [21]:
import os
import numpy as np
import matplotlib.pyplot as plt
os.environ["GRB_LICENSE_FILE"] = r"C:\Users\PC3\Desktop\gurobi.lic"

import gurobipy as gp
from gurobipy import GRB 


# Data Builder

In [8]:
from dataclasses import replace
import importlib
import uc_experiment_builders
importlib.reload(uc_experiment_builders)

from uc_experiment_builders import (
    load_network_uc_from_excel_fixed,   # or build_data_ieee_case14 if using IEEE
    diversify_demand_archetypes_preserve_C,
    scale_renewables,
    make_scenario_data,
)

# ============================================================
# 0) Build base_data ONCE (run this cell after defining paths)
# ============================================================
# Example: Gabriel Excel dataset
XLSX_PATH = r"C:\Users\tomas\Documents\GitHub\Counterfactual-Explanations-for-optimization-problems\Mixed\UC-Experiments\RedEjemploRiesgoConfiabilidad.xlsx"  # <-- put your path

base_data = load_network_uc_from_excel_fixed(
    xlsx_path=XLSX_PATH,
    wind_scenario=2,
    solar_scenario="Alto",
    curt_penalty=1000.0,
    allow_shifting=False,
    carbon_price=0.0,
    slack_bus=0,
    rep_params=None,                      # <-- you already have these in notebook
    emission_rates=None,     # <-- you already have these
)

# ============================================================
# 1) Scenario knobs
# ============================================================
ARCHETYPE_SEED = 7
REN_SCALE = 2.0

# ============================================================
# 2) Build scenario data from base_data
# ============================================================
data, types = make_scenario_data(base_data, archetype_seed=ARCHETYPE_SEED, ren_scale=REN_SCALE)

print("Scenario ready:")
print("  ARCHETYPE_SEED =", ARCHETYPE_SEED)
print("  REN_SCALE      =", REN_SCALE)
print("  demand shape   =", data.demand.shape)
print("  #gens, #rens   =", len(data.gens), len(data.rens))


Scenario ready:
  ARCHETYPE_SEED = 7
  REN_SCALE      = 2.0
  demand shape   = (14, 24)
  #gens, #rens   = 6 4


In [30]:
from dataclasses import replace
import importlib
import uc_experiment_builders
importlib.reload(uc_experiment_builders)

from uc_experiment_builders import (
    load_network_uc_from_excel_fixed,
    make_scenario_data,
)

XLSX_PATH = r"C:\Users\tomas\Documents\GitHub\Counterfactual-Explanations-for-optimization-problems\Mixed\UC-Experiments\RedEjemploRiesgoConfiabilidad.xlsx"

ARCHETYPE_SEED = 5
REN_SCALE = 2.0

def build_data(
    *,
    wind_scenario=2,
    solar_scenario="Alto",
    curt_penalty=1000.0,
    allow_shifting=False,
    carbon_price=0.0,
    slack_bus=0,
    rep_params=None,
    emission_rates=None,
    archetype_seed=ARCHETYPE_SEED,
    ren_scale=REN_SCALE,
):
    base_data = load_network_uc_from_excel_fixed(
        xlsx_path=XLSX_PATH,
        wind_scenario=wind_scenario,
        solar_scenario=solar_scenario,
        curt_penalty=curt_penalty,
        allow_shifting=allow_shifting,
        carbon_price=carbon_price,
        slack_bus=slack_bus,
        rep_params=rep_params,
        emission_rates=emission_rates,
    )

    data, types = make_scenario_data(base_data, archetype_seed=archetype_seed, ren_scale=ren_scale)
    return data, types


In [ ]:
import numpy as np

from uc_pipeline import NetworkUCData
from b3_ncxplain import (
    run_B3_ncxplain_shift_prices,
    run_E3_ncxplain_shift_and_curt_prices_emissions,
    foil_force_unit_commitment,
)
from b3_ncxplain import run_B3_ncxplain_shift_prices

data, types = build_data(allow_shifting=True)


out = run_B3_ncxplain_shift_prices(
    data=data,
    window_size=3,
    per_bus_neutrality=True,
    alpha=0.96,
    max_iters= 2000,
    verbose=True,
    output_flag_sp=1,
    output_flag_mp=0,
)

print(out["status"])

Set parameter OutputFlag to value 1
Set parameter IgnoreNames to value 0
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 5800H with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Academic license 2653481 - for non-commercial use only - registered to to___@ug.uchile.cl
Optimize a model with 4080 rows, 2496 columns and 9634 nonzeros (Min)
Model fingerprint: 0xf519ecc6
Model has 768 linear objective coefficients
Variable types: 2064 continuous, 432 integer (432 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+02]
  Objective range  [4e+01, 5e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e-01, 3e+02]
Presolve removed 3064 rows and 1242 columns
Presolve time: 0.01s
Presolved: 1016 rows, 1254 columns, 3858 nonzeros
Variable types: 844 continuous, 410 integer (410 binary)
Found heuristic solution: objective 2946581.3288
Found he

In [28]:

# ----------------------------
# Helpers: derive time series
# ----------------------------
def _get_T(data):
    return int(getattr(data, "T", data.demand.shape[1]))

def _renewable_avail_matrix(data):
    # (nR,T)
    if len(data.rens) == 0:
        return np.zeros((0, _get_T(data)))
    return np.vstack([np.asarray(r.avail, dtype=float).reshape(1, -1) for r in data.rens])

def _renewable_used_matrix(data, sol):
    # used = avail - curt  (nR,T)
    avail = _renewable_avail_matrix(data)
    curt  = np.asarray(sol.get("curt", np.zeros_like(avail)), dtype=float)
    return avail - curt

def _total_demand(data):
    # total raw demand (T,)
    return np.asarray(data.demand, dtype=float).sum(axis=0)

def _net_load_total(data, sol):
    # net load = demand + splus - sminus (T,)
    demand = np.asarray(data.demand, dtype=float)
    splus  = np.asarray(sol.get("splus", np.zeros_like(demand)), dtype=float)
    sminus = np.asarray(sol.get("sminus", np.zeros_like(demand)), dtype=float)
    return (demand + splus - sminus).sum(axis=0)

def _shed_total(sol):
    shed = np.asarray(sol.get("shed", 0.0), dtype=float)
    if np.isscalar(shed):
        return np.array([float(shed)])
    return shed.sum(axis=0)  # (T,)

def _served_total(data, sol):
    # served = net_load - shed
    return _net_load_total(data, sol) - _shed_total(sol)

def _gen_matrix(sol):
    # (nG,T)
    return np.asarray(sol["p"], dtype=float)

def _gen_total(sol):
    return _gen_matrix(sol).sum(axis=0)  # (T,)

def _renew_used_total(data, sol):
    return _renewable_used_matrix(data, sol).sum(axis=0)  # (T,)

def _curtail_total(sol):
    curt = np.asarray(sol.get("curt", 0.0), dtype=float)
    if np.isscalar(curt):
        return np.array([float(curt)])
    return curt.sum(axis=0)

def _emissions_total(data, sol):
    # total emissions per hour: sum_g er[g]*p[g,t]
    er = np.array([float(g.emission_rate) for g in data.gens], dtype=float).reshape(-1, 1)
    p  = _gen_matrix(sol)
    return (er * p).sum(axis=0)  # (T,)

def _voll_cost_total(data, sol):
    # shedding cost per hour = VOLL * shed_total
    # NOTE: in NetworkUCData you used VOLL (uppercase) in one file and voll in another
    voll = float(getattr(data, "VOLL", getattr(data, "voll", 20000.0)))
    return voll * _shed_total(sol)

def _curtail_cost_total(data, sol):
    # if you want the curtailment penalty cost per hour: sum_r curt_cost[r,t] * curt[r,t]
    if len(data.rens) == 0:
        return np.zeros(_get_T(data))
    curt_cost = np.vstack([np.asarray(r.curt_cost, dtype=float).reshape(1, -1) for r in data.rens])
    curt = np.asarray(sol["curt"], dtype=float)
    return (curt_cost * curt).sum(axis=0)

# ----------------------------
# Plotting functions
# ----------------------------
def plot_demand_served_shed(data, sol, title="Demand vs Served vs Shed"):
    T = _get_T(data)
    t = np.arange(T)

    demand_raw = _total_demand(data)
    net_load   = _net_load_total(data, sol)
    served     = _served_total(data, sol)
    shed       = _shed_total(sol)

    plt.figure()
    plt.plot(t, demand_raw, label="Raw demand (sum buses)")
    plt.plot(t, net_load,   label="Net load (demand+splus-sminus)")
    plt.plot(t, served,     label="Served load")
    plt.plot(t, shed,       label="Load shedding")
    plt.title(title)
    plt.xlabel("Hour")
    plt.ylabel("MW")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

def plot_generation_mix(data, sol, title="Generation mix (thermal + renew used)"):
    T = _get_T(data)
    t = np.arange(T)

    g_total = _gen_total(sol)
    r_used  = _renew_used_total(data, sol)
    served  = _served_total(data, sol)

    plt.figure()
    plt.plot(t, served, label="Served load")
    plt.plot(t, g_total, label="Thermal generation (sum p_g)")
    plt.plot(t, r_used, label="Renewables used (sum avail-curt)")
    plt.title(title)
    plt.xlabel("Hour")
    plt.ylabel("MW")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

def plot_curtailment(data, sol, title="Curtailment"):
    T = _get_T(data)
    t = np.arange(T)

    curt_total = _curtail_total(sol)
    avail_total = _renewable_avail_matrix(data).sum(axis=0) if len(data.rens) else np.zeros(T)
    used_total  = _renew_used_total(data, sol)

    plt.figure()
    plt.plot(t, avail_total, label="Renewables available")
    plt.plot(t, used_total,  label="Renewables used")
    plt.plot(t, curt_total,  label="Curtailment")
    plt.title(title)
    plt.xlabel("Hour")
    plt.ylabel("MW")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

def plot_emissions(data, sol, title="Emissions"):
    T = _get_T(data)
    t = np.arange(T)

    E = _emissions_total(data, sol)

    plt.figure()
    plt.plot(t, E, label="Hourly emissions (sum er_g * p_g)")
    plt.title(title)
    plt.xlabel("Hour")
    plt.ylabel("Emission units (er*MW)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

def plot_shedding_and_costs(data, sol, title="Shedding + VOLL cost + Curtailment cost"):
    T = _get_T(data)
    t = np.arange(T)

    shed_mw = _shed_total(sol)
    voll_cost = _voll_cost_total(data, sol)
    curt_cost = _curtail_cost_total(data, sol)

    plt.figure()
    plt.plot(t, shed_mw, label="Shed (MW)")
    plt.title(title + " (MW)")
    plt.xlabel("Hour")
    plt.ylabel("MW")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    plt.figure()
    plt.plot(t, voll_cost, label="VOLL cost per hour")
    plt.plot(t, curt_cost, label="Curtailment penalty per hour")
    plt.title(title + " (cost)")
    plt.xlabel("Hour")
    plt.ylabel("$")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

def plot_all_dashboard(data, sol, tag="Solution"):
    plot_demand_served_shed(data, sol, title=f"{tag}: Demand/Served/Shedding")
    plot_generation_mix(data, sol, title=f"{tag}: Generation mix")
    plot_curtailment(data, sol, title=f"{tag}: Curtailment")
    plot_emissions(data, sol, title=f"{tag}: Emissions")
    plot_shedding_and_costs(data, sol, title=f"{tag}: Reliability & cost")
    plt.show()

# ----------------------------
# Usage with your `out`
# ----------------------------
# out is result from run_B3_ncxplain_shift_prices or run_E3_...
# Example:
# plot_all_dashboard(data, out["sol_factual"], tag="Factual")
# plot_all_dashboard(data, out["sol_foil"],    tag="Foil")
# plot_all_dashboard(data, out["sol_opt"],     tag="Opt under new coeffs")


In [29]:
plot_all_dashboard(data, out["sol_factual"], tag="Factual")
plot_all_dashboard(data, out["sol_opt"],     tag="Opt under new coeffs")

KeyError: 'sol_factual'